In [42]:
import pandas as pd
import json

from sqlalchemy import create_engine
from pymongo import MongoClient

print("Бібліотеки успішно підключені!")

Бібліотеки успішно підключені!


In [43]:
engine = create_engine('sqlite:///case_03.db')

print("SQLite подключена!")

SQLite подключена!


In [44]:
client = MongoClient("mongodb://localhost:27017/", serverSelectionTimeoutMS=5000)
client.admin.command("ping")
db = client["case_03"]

print("MongoDB підключена!")

MongoDB підключена!


In [45]:
from pathlib import Path

required_files = [
    "suspects.csv",
    "visits_morning.csv",
    "visits_evening.csv",
    "visits_backup.csv",
    "statements.json",
    "alibi.json",
]

candidates = [Path("case_03"), Path(".")]
CASE_DIR = next(
    (folder for folder in candidates
     if all((folder / filename).exists() for filename in required_files)),
    None
)

if CASE_DIR is None:
    raise FileNotFoundError(
        "Усі вихідні файли не знайдені. Покладіть CSV та JSON поруч із ноутбуком "
        "або в папку case_03."
    )

print("Папка з вихідними даними:", CASE_DIR.resolve())

suspects = pd.read_csv(CASE_DIR / "suspects.csv")
visits_morning = pd.read_csv(CASE_DIR / "visits_morning.csv")
visits_evening = pd.read_csv(CASE_DIR / "visits_evening.csv")
visits_backup = pd.read_csv(CASE_DIR / "visits_backup.csv")

Папка з вихідними даними: C:\Users\wkrr1\Desktop\Lab1_Bologov_Maksym\case_03


In [46]:
suspects.to_sql(
    "suspects",
    engine,
    if_exists="replace",
    index=False
)

visits_morning.to_sql(
    "visits_morning",
    engine,
    if_exists="replace",
    index=False
)

visits_evening.to_sql(
    "visits_evening",
    engine,
    if_exists="replace",
    index=False
)

visits_backup.to_sql(
    "visits_backup",
    engine,
    if_exists="replace",
    index=False
)

print("Всі CSV завантажені в SQLite!")

Всі CSV завантажені в SQLite!


In [47]:
with open(CASE_DIR / "statements.json", "r", encoding="utf-8") as file:
    statements_data = json.load(file)

with open(CASE_DIR / "alibi.json", "r", encoding="utf-8") as file:
    alibi_data = json.load(file)

db.statements.delete_many({})
db.alibi.delete_many({})

if statements_data:
    db.statements.insert_many(statements_data)
if alibi_data:
    db.alibi.insert_many(alibi_data)

print(f"Завантажено свідчень: {len(statements_data)}")
print(f"Завантажено записів алібі: {len(alibi_data)}")

Завантажено свідчень: 8
Завантажено записів алібі: 6


In [48]:
suspects_sql = pd.read_sql_query(
    "SELECT * FROM suspects",
    engine
)

morning_sql = pd.read_sql_query(
    "SELECT * FROM visits_morning",
    engine
)

evening_sql = pd.read_sql_query(
    "SELECT * FROM visits_evening",
    engine
)

backup_sql = pd.read_sql_query(
    "SELECT * FROM visits_backup",
    engine
)

In [49]:
statements_df = pd.DataFrame(
    list(db.statements.find({}, {"_id": 0}))
)

alibi_df = pd.DataFrame(
    list(db.alibi.find({}, {"_id": 0}))
)

In [50]:
suspects_sql = pd.read_sql_query(
    "SELECT * FROM suspects",
    engine
)

morning_sql = pd.read_sql_query(
    "SELECT * FROM visits_morning",
    engine
)

evening_sql = pd.read_sql_query(
    "SELECT * FROM visits_evening",
    engine
)

backup_sql = pd.read_sql_query(
    "SELECT * FROM visits_backup",
    engine
)

In [51]:
statements_df = pd.DataFrame(
    list(db.statements.find({}, {"_id": 0}))
)

alibi_df = pd.DataFrame(
    list(db.alibi.find({}, {"_id": 0}))
)

In [52]:
display(suspects_sql.head())
display(morning_sql.head())
display(evening_sql.head())
display(backup_sql.head())

display(statements_df.head())
display(alibi_df.head())

,suspect_id,name,role
0,S01,Раїса Дубова,охоронець
1,S02,Захар Тесля,науковий співробітник
2,S03,Еліна Бортник,прибиральник
3,S04,Федір Швець,фотограф
4,S05,Устим Романко,доглядач


,visit_id,suspect_id,room,hour,sensor_id
0,V001,S08,Бароко,12,SEN-925
1,V004,S06,Зброярня,10,SEN-487
2,V007,S04,Антична,10,SEN-517
3,V014,S10,Єгипетська,11,SEN-726
4,V017,S05,Жовта,13,SEN-724


,visit_id,suspect_id,room,hour,sensor_id
0,V002,S05,Гобеленова,17,SEN-508
1,V003,S07,Бароко,19,SEN-741
2,V005,S10,Зброярня,15,SEN-497
3,V006,S05,Бароко,15,SEN-190
4,V008,S04,NaN,18,SEN-270


,visit_id,room
0,V008,Жовта
1,V010,Бароко
2,V021,Гобеленова
3,V029,Антична
4,V040,Бароко


,witness,saw_suspect_id,room,hour
0,студентка Гнатюк,S04,Дзеркальна,17
1,водій Юрченко,S05,Жовта,13
2,волонтер Сидір,S07,Бароко,18
3,турист Новак,S07,Бароко,19
4,касирка Олійник,S08,Бароко,12


,suspect_id,confirmed_by
0,S02,водій Юрченко
1,S03,студентка Гнатюк
2,S04,студентка Гнатюк
3,S05,студентка Гнатюк
4,S09,касирка Олійник


In [53]:
display(evening_sql.head())
display(backup_sql.head())

,visit_id,suspect_id,room,hour,sensor_id
0,V002,S05,Гобеленова,17,SEN-508
1,V003,S07,Бароко,19,SEN-741
2,V005,S10,Зброярня,15,SEN-497
3,V006,S05,Бароко,15,SEN-190
4,V008,S04,NaN,18,SEN-270


,visit_id,room
0,V008,Жовта
1,V010,Бароко
2,V021,Гобеленова
3,V029,Антична
4,V040,Бароко


In [54]:
evening_indexed = evening_sql.set_index("visit_id")

backup_indexed = backup_sql.set_index("visit_id")

In [55]:
evening_fixed = evening_indexed.combine_first(
    backup_indexed
)

In [56]:
evening_fixed = evening_fixed.loc[
    evening_indexed.index
]

In [57]:
evening_fixed["hour"] = evening_fixed["hour"].astype(int)

In [58]:
display(evening_fixed)

,suspect_id,room,hour,sensor_id
visit_id,,,,
V002,S05,Гобеленова,17,SEN-508
V003,S07,Бароко,19,SEN-741
V005,S10,Зброярня,15,SEN-497
V006,S05,Бароко,15,SEN-190
V008,S04,Жовта,18,SEN-270
V009,S01,Антична,19,SEN-986
V010,S08,Бароко,15,SEN-400
V011,S07,Бароко,18,SEN-580
V012,S02,Дзеркальна,20,SEN-197


In [59]:
visits_all = pd.concat(
    [
        evening_fixed.reset_index(),
        morning_sql
    ],
    ignore_index=True
)

visits_all = visits_all.drop(
    columns=["sensor_id"],
    errors="ignore"
)

display(visits_all.head())

,visit_id,suspect_id,room,hour
0,V002,S05,Гобеленова,17
1,V003,S07,Бароко,19
2,V005,S10,Зброярня,15
3,V006,S05,Бароко,15
4,V008,S04,Жовта,18


In [60]:
visits_all = visits_all.reset_index()

In [61]:
visits_all = visits_all.drop(
    columns=["sensor_id"],
    errors="ignore"
)

In [62]:
display(visits_all.head(20))

print("Всего записей:", len(visits_all))

,index,visit_id,suspect_id,room,hour
0,0,V002,S05,Гобеленова,17
1,1,V003,S07,Бароко,19
2,2,V005,S10,Зброярня,15
3,3,V006,S05,Бароко,15
4,4,V008,S04,Жовта,18
5,5,V009,S01,Антична,19
6,6,V010,S08,Бароко,15
7,7,V011,S07,Бароко,18
8,8,V012,S02,Дзеркальна,20
9,9,V013,S03,Гобеленова,15


Всего записей: 57


In [63]:
visits_named = pd.merge(
    visits_all,
    suspects_sql,
    on="suspect_id",
    how="left"
)

display(visits_named.head())

,index,visit_id,suspect_id,room,hour,name,role
0,0,V002,S05,Гобеленова,17,Устим Романко,доглядач
1,1,V003,S07,Бароко,19,Ярина Кушнір,екскурсовод
2,2,V005,S10,Зброярня,15,Денис Коваль,науковий співробітник
3,3,V006,S05,Бароко,15,Устим Романко,доглядач
4,4,V008,S04,Жовта,18,Федір Швець,фотограф


In [64]:
print(visits_named.columns.tolist())

['index', 'visit_id', 'suspect_id', 'room', 'hour', 'name', 'role']


In [65]:
display(statements_df.head())
display(suspects_sql.head())

,witness,saw_suspect_id,room,hour
0,студентка Гнатюк,S04,Дзеркальна,17
1,водій Юрченко,S05,Жовта,13
2,волонтер Сидір,S07,Бароко,18
3,турист Новак,S07,Бароко,19
4,касирка Олійник,S08,Бароко,12


,suspect_id,name,role
0,S01,Раїса Дубова,охоронець
1,S02,Захар Тесля,науковий співробітник
2,S03,Еліна Бортник,прибиральник
3,S04,Федір Швець,фотограф
4,S05,Устим Романко,доглядач


In [66]:
merge_inner = pd.merge(
    statements_df,
    suspects_sql,
    left_on="saw_suspect_id",
    right_on="suspect_id",
    how="inner"
)

In [67]:
merge_left = pd.merge(
    statements_df,
    suspects_sql,
    left_on="saw_suspect_id",
    right_on="suspect_id",
    how="left"
)

In [68]:
merge_right = pd.merge(
    statements_df,
    suspects_sql,
    left_on="saw_suspect_id",
    right_on="suspect_id",
    how="right"
)

In [69]:
merge_outer = pd.merge(
    statements_df,
    suspects_sql,
    left_on="saw_suspect_id",
    right_on="suspect_id",
    how="outer"
)

In [70]:
print("INNER:", len(merge_inner))
print("LEFT:", len(merge_left))
print("RIGHT:", len(merge_right))
print("OUTER:", len(merge_outer))

INNER: 6
LEFT: 8
RIGHT: 11
OUTER: 13


Результати об'єднання відрізняються, тому що кожен тип merge обробляє збіги по-різному.

INNER залишає лише збігаються записи.

LEFT зберігає всі записи з лівої таблиці.

RIGHT зберігає всі записи із правої таблиці.

OUTER об'єднує всі записи обох таблиць, навіть якщо збіги відсутні.

Свідки могли бачити сторонніх людей, яких немає у довіднику працівників. Також є працівники, про яких ніхто не свідчив. Тому кількість рядків після кожного типу об'єднання відрізняється.

In [71]:
def get_log():
    return visits_named.copy()

In [72]:
try:
    get_log().pivot(
        index="name",
        columns="room",
        values="visit_id"
    )
except ValueError as e:
    print("pivot отказался:", e)

pivot отказался: Index contains duplicate entries, cannot reshape


In [73]:
pivot_result = get_log().pivot_table(
    index="name",
    columns="room",
    values="visit_id",
    aggfunc="count",
    fill_value=0
)

display(pivot_result)

room,Єгипетська,Антична,Бароко,Вітражна,Гобеленова,Дзеркальна,Жовта,Зброярня
name,,,,,,,,
Денис Коваль,3,1,1,0,2,0,0,1
Еліна Бортник,0,1,1,0,1,1,1,1
Жанна Рудик,0,1,0,0,1,2,0,1
Захар Тесля,0,0,0,1,0,2,2,0
Оксана Литвин,1,0,0,0,0,1,0,2
Раїса Дубова,0,3,1,0,2,0,1,0
Устим Романко,0,0,1,0,3,0,1,1
Федір Швець,0,2,0,0,0,1,1,2
Христина Бойко,0,0,2,0,1,1,0,0


In [74]:
target_room = "Гобеленова"
target_hour = 17

suspects_at_scene = visits_named[
    (visits_named["room"] == target_room) &
    (visits_named["hour"] == target_hour)
]

display(
    suspects_at_scene[
        ["suspect_id", "name", "room", "hour"]
    ]
)

,suspect_id,name,room,hour
0,S05,Устим Романко,Гобеленова,17
11,S01,Раїса Дубова,Гобеленова,17
23,S10,Денис Коваль,Гобеленова,17


In [75]:
display(alibi_df)

,suspect_id,confirmed_by
0,S02,водій Юрченко
1,S03,студентка Гнатюк
2,S04,студентка Гнатюк
3,S05,студентка Гнатюк
4,S09,касирка Олійник
5,S10,водій Юрченко


In [76]:
print(alibi_df.columns.tolist())

['suspect_id', 'confirmed_by']


In [77]:
print(alibi_df.to_string(index=False))

suspect_id     confirmed_by
       S02    водій Юрченко
       S03 студентка Гнатюк
       S04 студентка Гнатюк
       S05 студентка Гнатюк
       S09  касирка Олійник
       S10    водій Юрченко


In [78]:
scene_check = pd.merge(
    suspects_at_scene,
    alibi_df,
    on="suspect_id",
    how="left"
)

In [79]:
culprit = scene_check[
    scene_check["confirmed_by"].isna()
]

In [80]:
display(scene_check)

,index,visit_id,suspect_id,room,hour,name,role,confirmed_by
0,0,V002,S05,Гобеленова,17,Устим Романко,доглядач,студентка Гнатюк
1,11,V016,S01,Гобеленова,17,Раїса Дубова,охоронець,NaN
2,23,V041,S10,Гобеленова,17,Денис Коваль,науковий співробітник,водій Юрченко


In [81]:
print("Винна:")
print(culprit["name"].to_string(index=False))

Винна:
Раїса Дубова


Під час аналізу журналу відвідувань було встановлено, що о 17:00 у Гобеленовій залі перебували троє працівників: Устим Романко, Раїса Дубова та Денис Коваль.

Після перевірки документів MongoDB встановлено, що Устим Романко та Денис Коваль мають підтверджене алібі.

Раїса Дубова не має запису про підтверджене алібі.

Отже, за умовами лабораторної роботи, підозрюваною, яка залишилася, є Раїса Дубова (S01).